In [41]:
import pandas as pd
import geopandas as gpd
from sklearn.linear_model import LinearRegression
import numpy as np

print("1. Fetching Data...")
# Here we will load your CBS/GGD Wijk and Buurt data, and your geographic boundaries.
# For this script, assume 'df' contains your merged historical data.
# df = pd.read_csv('ggd_cbs_merged_data.csv')
# geo_df = gpd.read_file('buurten.geojson')

# --- DUMMY DATA FOR ARCHITECTURE SETUP ---
# Replace this block with your actual data loading
df = pd.DataFrame({
    'buurt_code': ['BU001', 'BU002', 'BU003'],
    'health_risk_score': [45, 60, 30], # The GGD Health outcome we want to predict
    'temperature_pet': [32.5, 34.0, 30.1],
    'green_space_pct': [10, 5, 25],
    'old_housing_pct': [50, 80, 20],
    'vuln_demographics_pct': [15, 30, 10]
})
geo_df = gpd.GeoDataFrame({'buurt_code': ['BU001', 'BU002', 'BU003'], 'geometry': [None, None, None]})
# -----------------------------------------

print("2. Training the Prediction Model...")
# Define our features (X) and target (y)
features = ['temperature_pet', 'green_space_pct', 'old_housing_pct', 'vuln_demographics_pct']
X = df[features]
y = df['health_risk_score']

# Train the Multiple Linear Regression model
model = LinearRegression()
model.fit(X, y)

# Print out the learned weights (so you can explain them in your thesis!)
print("\nModel Coefficients (Impact on Health Risk):")
for feature, coef in zip(features, model.coef_):
    print(f" - {feature}: {coef:.4f}")

print("\n3. Generating Future & Policy Scenarios...")
# Create a copy of the dataframe to hold all our pre-calculated scenarios
scenarios_df = df[['buurt_code']].copy()

# SCENARIO 1: 2025 Baseline
scenarios_df['risk_2025_baseline'] = model.predict(X)

# SCENARIO 2: 2050 Climate Scenario (e.g., KNMI says +1.5 degrees)
X_2050 = X.copy()
X_2050['temperature_pet'] = X_2050['temperature_pet'] + 1.5
scenarios_df['risk_2050_baseline'] = model.predict(X_2050)

# SCENARIO 3: 2050 + Aggressive Greening Policy (+15% green space everywhere)
X_2050_green = X_2050.copy()
X_2050_green['green_space_pct'] = X_2050_green['green_space_pct'] + 15
scenarios_df['risk_2050_greening'] = model.predict(X_2050_green)

# SCENARIO 4: 2075 Climate Scenario (+3.0 degrees)
X_2075 = X.copy()
X_2075['temperature_pet'] = X_2075['temperature_pet'] + 3.0
scenarios_df['risk_2075_baseline'] = model.predict(X_2075)
print("4. Merging and Exporting...")
# Merge our scenarios back to the map shapes
final_map = geo_df.merge(scenarios_df, on='buurt_code')

# Save as a single GeoJSON file. This is the ONLY file our HTML frontend will need!
final_map.to_file('map_scenarios.geojson', driver='GeoJSON')
print("✅ Pre-generation complete. Ready for the frontend!")

1. Fetching Data...
2. Training the Prediction Model...

Model Coefficients (Impact on Health Risk):
 - temperature_pet: 0.0264
 - green_space_pct: -0.1347
 - old_housing_pct: 0.4077
 - vuln_demographics_pct: 0.1371

3. Generating Future & Policy Scenarios...
4. Merging and Exporting...
✅ Pre-generation complete. Ready for the frontend!


C:\Users\spygl\AppData\Roaming\Python\Python311\site-packages\pyogrio\geopandas.py:917: UserWarning: 'crs' was not provided.  The output dataset will not have projection information defined and may not be usable in other systems.
  write(


In [39]:
import geopandas as gpd
import pandas as pd
import numpy as np

print("1. Loading your REAL map shapes...")
# Load the actual map you used for your Streamlit app
geo_df = gpd.read_file('gemeenten.geojson')

# Extract whatever the municipality column is named (usually 'gemeentenaam')
# If your column is named differently, change it here!
region_col = 'gemeentenaam' 

print("2. Generating Test Health Scores...")
# Create a dataframe to hold our test scenarios
scenarios_df = pd.DataFrame({
    region_col: geo_df[region_col]
})

# Generate fake health risk scores (0-100) so we can test the HTML colors
# Baseline: Random score between 10 and 60
scenarios_df['risk_2025_baseline'] = np.random.randint(10, 60, size=len(geo_df))

# 2050 Baseline: Gets hotter, so scores go up (+20)
scenarios_df['risk_2050_baseline'] = scenarios_df['risk_2025_baseline'] + 20

# 2075 Baseline: Gets much hotter (+40)
scenarios_df['risk_2075_baseline'] = scenarios_df['risk_2025_baseline'] + 40

# 2050 Greening: Drops the 2050 risk by 15 points
scenarios_df['risk_2050_greening'] = scenarios_df['risk_2050_baseline'] - 15


print("3. Merging and Exporting...")
final_map = geo_df.merge(scenarios_df, on=region_col)

# THE FIX: Translate Dutch coordinates to standard GPS!
final_map = final_map.to_crs(epsg=4326)

# Overwrite the empty dummy file with this real, GPS-aligned map!
final_map.to_file('map_scenarios.geojson', driver='GeoJSON')
print("✅ Success! Hard refresh your HTML page.")

# Overwrite the empty dummy file with this real, colored map!
final_map.to_file('map_scenarios.geojson', driver='GeoJSON')
print("✅ Success! Refresh your HTML page.")

1. Loading your REAL map shapes...
2. Generating Test Health Scores...
3. Merging and Exporting...
✅ Success! Hard refresh your HTML page.
✅ Success! Refresh your HTML page.


In [3]:
import geopandas as gpd
import pandas as pd
import numpy as np

print("1. Loading Official Neighborhood shapes (GPKG)...")
# 1. LOAD THE GEOPACKAGE
# We specify layer='buurten' to ensure we only grab the 14,000 neighborhoods, 
# not the municipalities or districts that might also be inside the file.
geo_df = gpd.read_file('wijkenbuurten_2024.gpkg', layer='buurten') 

# 2. CHECK THE MERGE COLUMN
# CBS usually names this 'buurtcode'. If you get a KeyError later, it might be capitalized as 'BUURTCODE'.
region_col = 'buurtcode' 

print("2. Generating Neighborhood Scenarios...")
scenarios_df = pd.DataFrame({region_col: geo_df[region_col]})

# Generate fake health risk scores (0-100) for all 14,000+ neighborhoods
scenarios_df['risk_2025_baseline'] = np.random.randint(10, 60, size=len(geo_df))
scenarios_df['risk_2050_baseline'] = scenarios_df['risk_2025_baseline'] + 20
scenarios_df['risk_2075_baseline'] = scenarios_df['risk_2025_baseline'] + 40
scenarios_df['risk_2050_greening'] = scenarios_df['risk_2050_baseline'] - 15

print("3. Compressing, Merging, and Exporting...")
final_map = geo_df.merge(scenarios_df, on=region_col)

# 3. CRITICAL SURVIVAL STEP: Heavy Compression!
# The original CBS file is in Dutch coordinates (Amersfoort/RD), which uses meters.
# A tolerance of 100 simplifies borders by removing zig-zags smaller than 100 meters.
final_map['geometry'] = final_map['geometry'].simplify(tolerance=100)

# 4. Translate to standard GPS and export for the web browser
final_map = final_map.to_crs(epsg=4326)
final_map.to_file('map_scenarios.geojson', driver='GeoJSON')

print("✅ Neighborhood Map Compressed and Exported!")

1. Loading Official Neighborhood shapes (GPKG)...
2. Generating Neighborhood Scenarios...
3. Compressing, Merging, and Exporting...
✅ Neighborhood Map Compressed and Exported!


In [16]:
import pandas as pd
import numpy as np

# 1. LOAD DATA with specific 'dot' handling for CBS suppression [cite: 86]
print("1. Loading and Slimming Pillars...")
health_df = pd.read_csv('health_data.csv', delimiter=';', na_values='.')
cbs_raw = pd.read_excel('kwb2024.xlsx', na_values='.')
climate_df = pd.read_csv('Heat_values_1.csv', na_values='.')

# 2. SLIM CBS DATA (Prevents Fragmentation Warning)
# Selecting target features: Population, Elderly, Low Income, and Rental % [cite: 12, 15, 18]
kwb_cols = ['gwb_code_10', 'recs', 'regio', 'a_inw', 'a_65_oo', 'p_ink_li', 'p_huurw']
cbs_bu = cbs_raw[cbs_raw['recs'].astype(str).str.contains('Buurt', case=False)][kwb_cols].copy() # [cite: 160]

# 3. FILTER HEALTH DATA
# Isolate Buurt level[cite: 160], specific age group (18+), and most recent period.
health_bu = health_df[
    (health_df['SoortRegio_2'].astype(str).str.contains('Buurt', case=False)) &
    (health_df['Leeftijd'].astype(str).str.contains('18 jaar of ouder', case=False))
].copy()

# 4. STANDARDIZE THE GOLDEN KEY (Keep 'BU' + 8 digits) [cite: 164]
def clean_key(series):
    return series.astype(str).str.strip().str.upper()

health_bu['MergeKey'] = clean_key(health_bu['Codering_3'])
cbs_bu['MergeKey'] = clean_key(cbs_bu['gwb_code_10'])
climate_df['MergeKey'] = clean_key(climate_df['buurtcode'])

# 5. THE NEIGHBORHOOD MERGE
master_df = pd.merge(health_bu, cbs_bu, on='MergeKey', how='inner')
master_df = pd.merge(master_df, climate_df, on='MergeKey', how='inner')

# 6. CALCULATE VULNERABILITY SCORE
# Risk = 100% minus those with 'Good Health'
target_in = 'GoedErvarenGezondheid_4'
target_out = 'Vulnerability_Score'

master_df[target_in] = pd.to_numeric(master_df[target_in], errors='coerce')
master_df[target_out] = 100 - master_df[target_in]

# 7. FINAL CLEANING & FEATURE ENGINEERING
# Remove neighborhoods where health data was suppressed (the 'dot' issue) [cite: 86]
final_master = master_df.dropna(subset=[target_out, 'PET_gem']).copy()

# Elderly % = (Residents 65+ / Total Inhabitants) * 100 [cite: 211, 198]
final_master['p_elderly'] = (pd.to_numeric(final_master['a_65_oo'], errors='coerce') / 
                             pd.to_numeric(final_master['a_inw'], errors='coerce')) * 100

# Select professional headers for the Ministry dashboard
output_cols = ['MergeKey', 'regio', target_out, 'PET_gem', 'p_ink_li', 'p_elderly', 'p_huurw']
final_master = final_master[output_cols].rename(columns={'MergeKey': 'buurtcode', 'regio': 'buurtnaam'})

print(f"✅ SUCCESS: {len(final_master)} neighborhoods found with full health & climate data.")
final_master.to_csv('Neighborhood_Master_Data.csv', index=False)

1. Loading and Slimming Pillars...
✅ SUCCESS: 0 neighborhoods found with full health & climate data.


In [42]:
import pandas as pd
import numpy as np

print("1. Loading Raw Data Pillars...")
# Load with proper handling of missing values from the start
health_df = pd.read_csv('health_data.csv', delimiter=';', na_values=['.', ''])
cbs_df = pd.read_excel('kwb2024.xlsx', na_values=['.', '']) 
climate_df = pd.read_csv('Heat_values_1.csv', na_values=['.', '']) 

print("\n2. Standardize Keys FIRST (before any filtering)...")
health_df['MergeKey'] = health_df['Codering_3'].astype(str).str.strip().str.upper()
cbs_df['MergeKey'] = cbs_df['gwb_code_10'].astype(str).str.strip().str.upper()
climate_df['MergeKey'] = climate_df['buurtcode'].astype(str).str.strip().str.upper()

print("\n3. Filter for Neighborhoods (BU only)...")
health_bu = health_df[health_df['MergeKey'].str.startswith('BU')].copy()
cbs_bu = cbs_df[cbs_df['MergeKey'].str.startswith('BU')].copy()
climate_bu = climate_df[climate_df['MergeKey'].str.startswith('BU')].copy()

print(f"   Health neighborhoods: {len(health_bu)}")
print(f"   CBS neighborhoods: {len(cbs_bu)}")
print(f"   Climate neighborhoods: {len(climate_bu)}")

print("\n4. Merge carefully with both sides...")
# Start with CBS (which has income data) and merge health INTO it
master_df = cbs_bu.copy()
print(f"   Starting with CBS: {len(master_df)} rows")

# Merge health data - use left merge from CBS so we keep all CBS data
master_df = pd.merge(master_df, health_bu, on='MergeKey', how='left', suffixes=('_cbs', '_health'))
print(f"   After adding health: {len(master_df)} rows")

# Merge climate data
master_df = pd.merge(master_df, climate_bu, on='MergeKey', how='left')
print(f"   After adding climate: {len(master_df)} rows")

print("\n5. Feature Engineering & Data Cleaning...")

# Clean data format issues (handle Dutch decimals and CBS suppression)
print("\n  Cleaning data format issues...")
for col in ['GoedErvarenGezondheid_4', 'p_ink_li', 'HeelVeelStressAfg4Weken_17']:
    if col in master_df.columns:
        # Replace '.' with empty string, then comma with period
        master_df[col] = master_df[col].astype(str).str.replace('.', '', regex=False).str.replace(',', '.')
        # Convert to numeric (will become NaN for empty strings)
        master_df[col] = pd.to_numeric(master_df[col], errors='coerce')

# Convert other numeric columns
for col in ['PET_gem', 'p_huurw', 'a_65_oo', 'a_inw', 'a_00_14', 'p_bj_me10']:
    if col in master_df.columns:
        master_df[col] = pd.to_numeric(master_df[col], errors='coerce')

# CALCULATE NEW FEATURES
# p_elderly (age 65+)
if 'a_65_oo' in master_df.columns and 'a_inw' in master_df.columns:
    master_df['p_elderly'] = (master_df['a_65_oo'] / master_df['a_inw']) * 100

# p_children (age 0-14) - NEW
if 'a_00_14' in master_df.columns and 'a_inw' in master_df.columns:
    master_df['p_children'] = (master_df['a_00_14'] / master_df['a_inw']) * 100

# p_bj_me10 (old buildings %) - already exists, just ensure it's numeric
# p_stress - already exists as HeelVeelStressAfg4Weken_17

# Handle duplicate neighborhoods (aggregate by taking mean)
print("  Removing duplicates...")
print(f"   Rows before dedup: {len(master_df)}")
final_master = master_df[[
    'MergeKey', 
    'GoedErvarenGezondheid_4', 
    'PET_gem', 
    'p_ink_li', 
    'p_elderly', 
    'p_huurw',
    'p_children',
    'p_bj_me10',
    'HeelVeelStressAfg4Weken_17'
]].copy()

# Group by neighborhood and take the mean (handles duplicate rows from health data periods)
final_master = final_master.groupby('MergeKey', as_index=False).mean(numeric_only=True)
print(f"   Rows after dedup: {len(final_master)}")

# Drop rows with missing critical values
final_master = final_master.dropna(subset=['GoedErvarenGezondheid_4'])

print(f"\n✅ FINAL DATASET: {len(final_master)} neighborhoods ready for modeling!")
print(f"\n  Data quality in final dataset:")
for col in ['GoedErvarenGezondheid_4', 'PET_gem', 'p_ink_li', 'p_elderly', 'p_huurw', 'p_children', 'p_bj_me10', 'HeelVeelStressAfg4Weken_17']:
    if col in final_master.columns:
        valid = final_master[col].notna().sum()
        pct = 100 * valid / len(final_master) if len(final_master) > 0 else 0
        mn = final_master[col].min()
        mx = final_master[col].max()
        print(f"    - {col}: {valid}/{len(final_master)} ({pct:.0f}%) | Range: {mn:.1f} to {mx:.1f}")

print(f"\nSample (first 5 rows):")
print(final_master.head())

print(f"\n💾 Saving to CSV...")
final_master.to_csv('Neighborhood_Master_Data.csv', index=False)


1. Loading Raw Data Pillars...

2. Standardize Keys FIRST (before any filtering)...

3. Filter for Neighborhoods (BU only)...
   Health neighborhoods: 43722
   CBS neighborhoods: 14574
   Climate neighborhoods: 14574

4. Merge carefully with both sides...
   Starting with CBS: 14574 rows
   After adding health: 43722 rows
   After adding climate: 43722 rows

5. Feature Engineering & Data Cleaning...

  Cleaning data format issues...


C:\Users\spygl\AppData\Local\Temp\ipykernel_23756\2253827754.py:12: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  cbs_df['MergeKey'] = cbs_df['gwb_code_10'].astype(str).str.strip().str.upper()


  Removing duplicates...
   Rows before dedup: 43722
   Rows after dedup: 14574

✅ FINAL DATASET: 11734 neighborhoods ready for modeling!

  Data quality in final dataset:
    - GoedErvarenGezondheid_4: 11734/11734 (100%) | Range: 436.7 to 910.7
    - PET_gem: 11644/11734 (99%) | Range: 35.0 to 50.0
    - p_ink_li: 11678/11734 (100%) | Range: 7.1 to 95.1
    - p_elderly: 11734/11734 (100%) | Range: 0.0 to 100.0
    - p_huurw: 11689/11734 (100%) | Range: 0.0 to 100.0
    - p_children: 11734/11734 (100%) | Range: 0.0 to 39.4
    - p_bj_me10: 11689/11734 (100%) | Range: 0.0 to 100.0
    - HeelVeelStressAfg4Weken_17: 11734/11734 (100%) | Range: 76.3 to 459.0

Sample (first 5 rows):
     MergeKey  GoedErvarenGezondheid_4  PET_gem  p_ink_li  p_elderly  p_huurw  \
0  BU00140000               813.000000     44.0      63.8   6.954689     87.0   
1  BU00140001               809.000000     44.0      65.5   7.383513     88.0   
2  BU00140002               777.333333     45.0      61.9   8.750000  

In [43]:
# 1. IMPORTS
from sklearn.linear_model import LinearRegression
import pandas as pd
import geopandas as gpd
import numpy as np

# 2. LOAD YOUR MASTER DATA
df = pd.read_csv('Neighborhood_Master_Data.csv')

# Remove any rows with missing values in features
df = df.dropna(subset=['GoedErvarenGezondheid_4', 'PET_gem', 'p_ink_li', 'p_elderly', 'p_huurw', 'p_children', 'p_bj_me10', 'HeelVeelStressAfg4Weken_17'])
print(f"Neighborhoods after removing NaNs: {len(df)}")

# 3. DEFINE THE MODEL (NOW WITH 7 FEATURES)
# Target: GoedErvarenGezondheid_4 (Good Health Perception - INVERSE of vulnerability)
# Higher values = better health = LOWER vulnerability score needed
# 
# Features (7 total):
# Original 4:
#   - PET_gem (Perceived Temperature - Heat exposure)
#   - p_ink_li (Low Income %)
#   - p_elderly (Age 65+ %)
#   - p_huurw (Rental Housing %)
# New 3:
#   - p_children (Age 0-14 %) - Vulnerable age group
#   - p_bj_me10 (Old Buildings % built before 2010) - Poor insulation/cooling
#   - HeelVeelStressAfg4Weken_17 (High Stress %) - Mental health vulnerability

X = df[['PET_gem', 'p_ink_li', 'p_elderly', 'p_huurw', 'p_children', 'p_bj_me10', 'HeelVeelStressAfg4Weken_17']]
y = 100 - df['GoedErvarenGezondheid_4']  # Convert: invert so higher = worse health outcome

# Train the prediction model
model = LinearRegression()
model.fit(X, y)

print("✅ Model Trained Successfully on " + str(len(df)) + " neighborhoods!")
print(f"\n🚨 Model Coefficients (Impact on Health Vulnerability):")
print(f"Intercept (Baseline Risk): {model.intercept_:.4f}")
for feature, coef in zip(X.columns, model.coef_):
    direction = "increases" if coef > 0 else "decreases"
    abs_coef = abs(coef)
    print(f" - {feature}: {coef:+.6f} ({direction} risk) | Impact: {abs_coef:.2f}")

# Calculate R-squared for model quality check
from sklearn.metrics import r2_score
y_pred = model.predict(X)
r2 = r2_score(y, y_pred)
print(f"\n📊 Model Quality (R²): {r2:.4f}")

# 4. PRE-GENERATE SCENARIOS
# Baseline 2025
df['risk_2025_baseline'] = model.predict(X)

# 2050 Scenario: Increase PET temperature by 2 degrees
X_2050 = X.copy()
X_2050['PET_gem'] += 2.0
df['risk_2050_baseline'] = model.predict(X_2050)

# 2075 Scenario: Increase PET temperature by 4 degrees
X_2075 = X.copy()
X_2075['PET_gem'] += 4.0
df['risk_2075_baseline'] = model.predict(X_2075)

# Policy Simulation: Multiple interventions
# - Reduce rental housing dependency (new housing/ownership programs)
# - Reduce stress through community programs (stress drops 15%)
X_policy = X_2050.copy()
X_policy['p_huurw'] -= 15  
X_policy['p_huurw'] = X_policy['p_huurw'].clip(lower=0)
X_policy['HeelVeelStressAfg4Weken_17'] -= 15  
X_policy['HeelVeelStressAfg4Weken_17'] = X_policy['HeelVeelStressAfg4Weken_17'].clip(lower=0)
df['risk_2050_greening'] = model.predict(X_policy)

print(f"\n📊 Raw Risk Score Statistics (before normalization):")
for col in ['risk_2025_baseline', 'risk_2050_baseline', 'risk_2075_baseline', 'risk_2050_greening']:
    mn = df[col].min()
    mx = df[col].max()
    avg = df[col].mean()
    print(f"   {col}: Min={mn:.1f}, Max={mx:.1f}, Avg={avg:.1f}")

# NORMALIZE TO 0-100 SCALE FOR VISUALIZATION
# Find global min/max across all scenarios for consistent scaling
all_risks = pd.concat([
    df['risk_2025_baseline'], 
    df['risk_2050_baseline'], 
    df['risk_2075_baseline'],
    df['risk_2050_greening']
])
global_min = all_risks.min()
global_max = all_risks.max()

print(f"\n📏 Normalization range: {global_min:.1f} to {global_max:.1f}")

# Normalize all risk columns to 0-100 scale
for col in ['risk_2025_baseline', 'risk_2050_baseline', 'risk_2075_baseline', 'risk_2050_greening']:
    df[col] = 100 * (df[col] - global_min) / (global_max - global_min)

print(f"\n✅ Normalized Risk Score Statistics (0-100 scale):")
for col in ['risk_2025_baseline', 'risk_2050_baseline', 'risk_2075_baseline', 'risk_2050_greening']:
    mn = df[col].min()
    mx = df[col].max()
    avg = df[col].mean()
    print(f"   {col}: Min={mn:.1f}, Max={mx:.1f}, Avg={avg:.1f}")

# 5. CONNECT TO THE NEIGHBORHOOD MAP
geo_df = gpd.read_file('wijkenbuurten_2024.gpkg', layer='buurten')
print(f"\n🗺️ Loaded {len(geo_df)} neighborhoods from GeoPackage")

# Key step: Standardize the buurtcode in the geopackage to match our MergeKey
geo_df['buurtcode_clean'] = geo_df['buurtcode'].astype(str).str.strip().str.upper()
df['MergeKey_clean'] = df['MergeKey'].astype(str).str.strip().str.upper()

# Merge predictions into map
print(f"   Merging predictions...")
final_map = geo_df.merge(df[['MergeKey_clean', 'risk_2025_baseline', 'risk_2050_baseline', 'risk_2075_baseline', 'risk_2050_greening']], 
                          left_on='buurtcode_clean', right_on='MergeKey_clean', how='left')

# Check merge success
merged_count = final_map['risk_2025_baseline'].notna().sum()
print(f"   ✅ Successfully merged {merged_count} neighborhoods with predictions")

# 6. EXPORT FOR WEB - with optimizations
final_map = final_map.to_crs(epsg=4326)
final_map['geometry'] = final_map['geometry'].simplify(tolerance=0.0001)

# Select only essential columns for the GeoJSON (smaller file size)
export_cols = ['buurtcode', 'risk_2025_baseline', 'risk_2050_baseline', 'risk_2075_baseline', 'risk_2050_greening', 'geometry']
final_map_export = final_map[[c for c in export_cols if c in final_map.columns]].copy()

final_map_export.to_file('map_scenarios.geojson', driver='GeoJSON')

print(f"\n✅ GeoJSON exported: {merged_count} neighborhoods with normalized risk predictions (0-100 scale)")
print("   Ready for web dashboard!")


Neighborhoods after removing NaNs: 11554
✅ Model Trained Successfully on 11554 neighborhoods!

🚨 Model Coefficients (Impact on Health Vulnerability):
Intercept (Baseline Risk): -1268.3722
 - PET_gem: +4.174359 (increases risk) | Impact: 4.17
 - p_ink_li: +2.976344 (increases risk) | Impact: 2.98
 - p_elderly: +3.437598 (increases risk) | Impact: 3.44
 - p_huurw: +0.608436 (increases risk) | Impact: 0.61
 - p_children: +2.891703 (increases risk) | Impact: 2.89
 - p_bj_me10: +0.655575 (increases risk) | Impact: 0.66
 - HeelVeelStressAfg4Weken_17: +0.661834 (increases risk) | Impact: 0.66

📊 Model Quality (R²): 0.7220

📊 Raw Risk Score Statistics (before normalization):
   risk_2025_baseline: Min=-799.6, Max=-374.8, Avg=-645.7
   risk_2050_baseline: Min=-791.2, Max=-366.4, Avg=-637.3
   risk_2075_baseline: Min=-782.9, Max=-358.1, Avg=-629.0
   risk_2050_greening: Min=-803.0, Max=-385.5, Avg=-655.5

📏 Normalization range: -803.0 to -358.1

✅ Normalized Risk Score Statistics (0-100 scale):


In [36]:


# === MODEL SUMMARY FOR YOUR CAPSTONE THESIS ===
print("\n" + "="*70)
print("HEAT VULNERABILITY MODEL - NEIGHBORHOOD ANALYSIS")
print("="*70)

print("\n# Define Features (X) and Target (y)")
print("X = final_master[['PET_gem', 'p_ink_li', 'p_elderly', 'p_huurw']]")
print("y = 100 - final_master['GoedErvarenGezondheid_4']  # (Inverted: lower good health = higher vulnerability)")

print("\n# Train the model")
print("model = LinearRegression()")
print("model.fit(X, y)")

print("\n🚨 Model Trained on Real Neighborhood Data")
print(f"Intercept (Baseline Risk): {model.intercept_:.4f}")
for feature, coef in zip(X.columns, model.coef_):
    print(f" - {feature}: {coef:+.4f}")

print("\n" + "="*70)
print("INTERPRETATION:")
print("="*70)
print("""
Each coefficient shows how much vulnerability INCREASES for every 1-unit increase:

✓ PET_gem (+4.46): Every +1°C perceived temperature = +4.46% vulnerability
✓ p_ink_li (+3.25): Every +1% low-income residents = +3.25% vulnerability  
✓ p_elderly (+1.43): Every +1% elderly residents = +1.43% vulnerability
✓ p_huurw (+1.40): Every +1% rental housing = +1.40% vulnerability

This validates that heat vulnerability is driven by BOTH climate AND socio-economic factors.
""")


HEAT VULNERABILITY MODEL - NEIGHBORHOOD ANALYSIS

# Define Features (X) and Target (y)
X = final_master[['PET_gem', 'p_ink_li', 'p_elderly', 'p_huurw']]
y = 100 - final_master['GoedErvarenGezondheid_4']  # (Inverted: lower good health = higher vulnerability)

# Train the model
model = LinearRegression()
model.fit(X, y)

🚨 Model Trained on Real Neighborhood Data
Intercept (Baseline Risk): -1047.8956
 - PET_gem: +4.4612
 - p_ink_li: +3.2462
 - p_elderly: +1.4273
 - p_huurw: +1.4013

INTERPRETATION:

Each coefficient shows how much vulnerability INCREASES for every 1-unit increase:

✓ PET_gem (+4.46): Every +1°C perceived temperature = +4.46% vulnerability
✓ p_ink_li (+3.25): Every +1% low-income residents = +3.25% vulnerability  
✓ p_elderly (+1.43): Every +1% elderly residents = +1.43% vulnerability
✓ p_huurw (+1.40): Every +1% rental housing = +1.40% vulnerability

This validates that heat vulnerability is driven by BOTH climate AND socio-economic factors.



In [18]:

# DIAGNOSTIC: What keys actually exist in each dataset?
print("🔍 BEFORE MERGE DIAGNOSTIC:\n")

print(f"Health data loaded: {len(health_df)} rows")
print(f"Sample health keys: {health_df['Codering_3'].head().tolist()}")
print(f"Health key format example: {repr(health_df['Codering_3'].iloc[0])}")

print(f"\nCBS data loaded: {len(cbs_df)} rows")
print(f"Sample CBS keys: {cbs_df['gwb_code_10'].head().tolist()}")
print(f"CBS key format example: {repr(cbs_df['gwb_code_10'].iloc[0])}")

print(f"\nClimate data loaded: {len(climate_df)} rows")
print(f"Sample climate keys: {climate_df['buurtcode'].head().tolist()}")
print(f"Climate key format example: {repr(climate_df['buurtcode'].iloc[0])}")

# After uppercase/strip
print("\n" + "="*60)
print("AFTER STANDARDIZATION:\n")

health_df['MergeKey'] = health_df['Codering_3'].astype(str).str.strip().str.upper()
cbs_df['MergeKey'] = cbs_df['gwb_code_10'].astype(str).str.strip().str.upper()
climate_df['MergeKey'] = climate_df['buurtcode'].astype(str).str.strip().str.upper()

print(f"Health BU keys: {health_df[health_df['MergeKey'].str.startswith('BU')].shape[0]}")
print(f"CBS BU keys: {cbs_df[cbs_df['MergeKey'].str.startswith('BU')].shape[0]}")
print(f"Climate BU keys: {climate_df[climate_df['MergeKey'].str.startswith('BU')].shape[0]}")

# Check for overlap
health_bu_set = set(health_df[health_df['MergeKey'].str.startswith('BU')]['MergeKey'].unique())
cbs_bu_set = set(cbs_df[cbs_df['MergeKey'].str.startswith('BU')]['MergeKey'].unique())
climate_bu_set = set(climate_df[climate_df['MergeKey'].str.startswith('BU')]['MergeKey'].unique())

print(f"\n⚠️ Overlap between Health & CBS: {len(health_bu_set & cbs_bu_set)} neighborhoods")
print(f"⚠️ Overlap between (Health∩CBS) & Climate: {len((health_bu_set & cbs_bu_set) & climate_bu_set)} neighborhoods")

if health_bu_set & cbs_bu_set:
    print(f"\nSample overlapping keys:")
    for key in list(health_bu_set & cbs_bu_set)[:3]:
        print(f"  - {key}")


🔍 BEFORE MERGE DIAGNOSTIC:

Health data loaded: 54930 rows
Sample health keys: ['NL01      ', 'GM1680    ', 'WK168000  ', 'BU16800000', 'BU16800009']
Health key format example: 'NL01      '

CBS data loaded: 18310 rows
Sample CBS keys: ['NL00', 'GM0014', 'WK001400', 'BU00140000', 'BU00140001']
CBS key format example: 'NL00'

Climate data loaded: 14574 rows
Sample climate keys: ['BU00140000', 'BU00140001', 'BU00140002', 'BU00140003', 'BU00140004']
Climate key format example: 'BU00140000'

AFTER STANDARDIZATION:

Health BU keys: 43722
CBS BU keys: 14574
Climate BU keys: 14574

⚠️ Overlap between Health & CBS: 14574 neighborhoods
⚠️ Overlap between (Health∩CBS) & Climate: 14574 neighborhoods

Sample overlapping keys:
  - BU03970006
  - BU04515000
  - BU0363NL01


In [10]:
# Check how much data survived the numeric conversion
print("📊 Data Reality Check:")
check_cols = ['BrozeGezondheid_18', 'PET_gem', 'p_ink_li', 'p_huurw']
for col in check_cols:
    valid_count = pd.to_numeric(master_df[col], errors='coerce').notna().sum()
    print(f" - {col}: {valid_count} valid rows found out of {len(master_df)}")

📊 Data Reality Check:
 - BrozeGezondheid_18: 0 valid rows found out of 0
 - PET_gem: 0 valid rows found out of 0
 - p_ink_li: 0 valid rows found out of 0
 - p_huurw: 0 valid rows found out of 0
